In [5]:
import numpy as np
import matplotlib.pyplot as plt 
import skimage.io as io 
import scipy.ndimage as ndi 
import skimage.color as clr 
from sklearn.cluster import k_means
from skimage.exposure import equalize_hist
import skimage.morphology as morph

path = 'C:\\Users\\rocco\\Documents\\università\\ESM\\laboratorio\\Immagini\\'

## Operazioni Morfologiche

La morfologia matematica è un insieme di strumenti per l'elaborazione di immagini binarie o immagini su scala di grigio. Di solito le immagini binarie escono fuori da operazioni di thresholding. Ci permettono di modificare la forma di oggetti rilevati dalla segmentazione. 
Risolveremo questi problemi:
1. Eliminazione rumore e oggetti indesiderati;
2. Separare componenti sovrapposte;
3. Riconoscere forme simili.

Tutti gli operatori di morfologia matematica si basano sul confronto locale delle forme e strutture dell'immagine con una forma di riferimento: l'__elemento strutturante__.
La scelta dell'elemento strutturante diepnde dal tipo di informazione che si vuole mettere in evidenza. Tipicamente, l'elemento strutturante è __simmetrico__, __connesso__ e __convesso__. 
Il confronto locale avviene un'immagine binaria e una maschera binaria, attraverso una finestra scorrevole. La scelta dell'elemento strutturante corrisponde alla scelta di una maschera. 
Gli elementi strutturanti più comuni sono la croce, il quadrato e la T.
- __DILATAZIONE__ -> Pixel in uscita alto se c'è almeno un valore alto in output dalla finstra scorrevole. Riempie i buchi di dimensione più piccola dell'elemento strutturante, allarga le estremità delle forme, riempie i passaggi stretti e salda oggetti a distanza inferiore alla taglia dell'elemento strutturante;
- __EROSIONE__ -> Pixel in uscita alto se tutti i valori sono alti in output dalla finestra scorrevole. Elimina le componenti connesse di dimensione più piccola dell'elemento strutturante, cancella le estremità sottili, allarga buchi e passaggi e separa oggetti connessi da un ponte stretto;

L'__APERTURA MORFOLOGICA__ è costituita da un'erosione seguita da una dilatazione effettuata mediante lo stesso elemento stutturante. L'effetto è quello di preservare il più possibile le regioni di forma simile all'elemento strutturante, eliminando quelle differenti. Si tratta di un filtro di smoothing morfologico, il cui effetto è determinato dalla forma e dimensioni dell'elemento strutturante.
Si può usare per _trovare_ elementi specifici, sulla base della forma dell'elemento strutturante.

In [ ]:
# definizione maschere

b1 = morph.disk(7)
b2 = morph.diamond(7)
b3 = morph.octagon(7,7)
b4 = morph.ellipse(7,7)

plt.figure(1)
plt.subplot(1,4,1)
plt.imshow(b1, clim=[0,1], cmap='gray')
plt.subplot(1,4,2)
plt.imshow(b2, clim=[0,1], cmap='gray')
plt.subplot(1,4,3)
plt.imshow(b3, clim=[0,1], cmap='gray')
plt.subplot(1,4,4)
plt.imshow(b4, clim=[0,1], cmap='gray')

plt.show()

## Estrazione dei bordi

L'estrazione dei bordi si può ottenere combinando due operazioni: La sottrazione tra un'immagine e l'erosione dell'immagine stessa. La dimensione del bordo è determinata dalla dimensione dell'elemento strutturante.

In [ ]:
# estrazione dei bordi

im = path + 'volto_bool.tif'
x = np.float32(io.imread(im))

k = 3
s = morph.footprint_rectangle((k,k))
y = x - morph.erosion(x, s)


plt.figure(1)
plt.imshow(x, clim=[0,1], cmap='gray')
plt.title('input')
plt.figure(2)
plt.imshow(y, clim=[0,1], cmap='gray')
plt.title('output')
plt.show()

## Hit or Miss

Strumento di base per la shape detection nelle immagini binarie. Vanno definite due maschere, una di background e una di foreground. In quella di foreground, sono alti i pixel dove voglio in output i pixel alti, mentre in quella di background, sono alti i pixel dove voglio in output pixel bassi. Chiaramente, le due mappe sono mutualmente esclusive sulle posizioni.
Il risultato della hit or miss è = (A _erosione_ B1) & (A* _erosione_ B2).